# CONEXIÓN A LA API DE CLINICALTRIALS.GOV: ENSAYOS CLÍNICOS EN GLIOBLASTOMA

Aitor Madoz González. Proyecto del Máster en Big Data Science e Inteligencia Artificial (Universidad de Navarra), asignatura de Técnicas de Recogida de Datos.

El glioblastoma es el tumor cerebral maligno más frecuente y agresivo en adultos. Su tratamiento estándar apenas ha cambiado en casi veinte años y la supervivencia media ronda los 15 meses, por lo que es una enfermedad con una gran necesidad médica no cubierta en la que muchas farmacéuticas investigan nuevos tratamientos. Para una farmacéutica que quisiera entrar en este campo, el primer paso sería conocer qué ensayos clínicos existen, quién los patrocina, en qué fase están, qué fármacos prueban y cuántos pacientes incluyen. En este proyecto nos conectamos a la API de ClinicalTrials.gov, el registro oficial de ensayos clínicos de Estados Unidos, que recoge estudios de todo el mundo, y construimos un dataframe con todos los ensayos de glioblastoma. No es web scraping: no leemos el HTML de ninguna página, sino que pedimos los datos directamente a la API, que los devuelve en formato JSON.

Primero hicimos una petición pequeña, de solo dos estudios, para comprobar que la API respondía y ver cómo estaban organizados los datos, siguiendo la documentación oficial de la API. Con esa estructura escribimos una función que extrae de cada estudio el identificador, el título, el estado, la fase, el patrocinador y su tipo, los fármacos o tratamientos probados, el número de pacientes, el diseño del ensayo, las fechas de inicio y de fin, los países, el número de centros y si tiene resultados publicados. Como la API entrega los resultados por páginas, hicimos un bucle que va pidiendo página tras página mediante el marcador nextPageToken hasta descargar todos los estudios, 2288 en total. Después limpiamos la tabla traduciendo los códigos de la API, convirtiendo las fechas a formato fecha y ordenando los ensayos de la fase más avanzada a la menos avanzada y, dentro de cada fase, de más a menos pacientes. Por último mostramos los cincuenta primeros con el título coloreado según cómo ha ido el ensayo: verde si está completado, amarillo si está en proceso y rojo si se terminó antes de tiempo, se suspendió o se retiró. Que un ensayo esté completado significa que terminó según lo previsto, no que el fármaco funcionara.

Los principales problemas fueron tres. No todos los estudios tienen todos los campos, por ejemplo los observacionales no tienen fase, así que el código tenía que tolerar datos ausentes y dejarlos como nulos sin romperse. Existen dos fechas de finalización distintas y tuvimos que elegir la que indica cuándo habrá resultados. Y al ordenar por fecha de inicio casi todos los ensayos mostrados aparecían en proceso, porque un ensayo de glioblastoma tarda años en terminar, por lo que cambiamos el orden a fase y tamaño.

El resultado es un dataframe con los 2288 ensayos de glioblastoma registrados y dieciséis columnas. Cambiando el nombre de la enfermedad en los parámetros, el mismo notebook sirve para obtener los ensayos de cualquier otra enfermedad.

### IMPORTAMOS LAS LIBRERÍAS Y DEFINIMOS LOS PARÁMETROS

In [1]:
import json
import time

import pandas as pd
import requests

URL_API = "https://clinicaltrials.gov/api/v2/studies"
ENFERMEDAD = "glioblastoma"
ESTUDIOS_POR_PAGINA = 100
FILAS_A_MOSTRAR = 50

### PRIMERA CONEXIÓN A LA API

In [2]:
respuesta = requests.get(URL_API, params={"query.cond": ENFERMEDAD, "pageSize": 2, "countTotal": "true"})
print(respuesta.status_code)

datos = respuesta.json()
print(datos.keys())
print(datos.get("totalCount"))

200
dict_keys(['totalCount', 'studies', 'nextPageToken'])
2288


### VEMOS CÓMO ES UN ESTUDIO POR DENTRO

In [3]:
estudio = datos["studies"][0]
print(estudio.keys())
print(estudio["protocolSection"].keys())

for modulo in ["identificationModule", "statusModule", "sponsorCollaboratorsModule", "designModule"]:
    print("\n=====", modulo)
    print(json.dumps(estudio["protocolSection"].get(modulo), indent=2)[:1500])

dict_keys(['protocolSection', 'resultsSection', 'derivedSection', 'hasResults'])
dict_keys(['identificationModule', 'statusModule', 'sponsorCollaboratorsModule', 'descriptionModule', 'conditionsModule', 'designModule', 'armsInterventionsModule', 'outcomesModule', 'eligibilityModule', 'contactsLocationsModule', 'referencesModule'])

===== identificationModule
{
  "nctId": "NCT00875433",
  "orgStudyIdInfo": {
    "id": "1200.24"
  },
  "secondaryIdInfos": [
    {
      "id": "2008-006288-36",
      "type": "EUDRACT_NUMBER",
      "domain": "EudraCT"
    },
    {
      "id": "2008",
      "type": "NIH",
      "link": "https://reporter.nih.gov/quickSearch/2008"
    }
  ],
  "organization": {
    "fullName": "Boehringer Ingelheim",
    "class": "INDUSTRY"
  },
  "briefTitle": "Afatinib (BIBW 2992) QTcF Trial in Patients With Relapsed or Refractory Solid Tumours",
  "officialTitle": "Phase II Open Label Trial to Assess the Efficacy and the Impact on QTcF of Continuous Oral BIBW 2992 at a Dai

### FUNCIÓN PARA SACAR LOS DATOS DE CADA ESTUDIO

In [4]:
def extraer_fila(estudio):
    protocolo = estudio["protocolSection"]
    identificacion = protocolo.get("identificationModule", {})
    estado = protocolo.get("statusModule", {})
    patrocinador = protocolo.get("sponsorCollaboratorsModule", {}).get("leadSponsor", {})
    diseno = protocolo.get("designModule", {})
    info_diseno = diseno.get("designInfo", {})
    intervenciones = protocolo.get("armsInterventionsModule", {}).get("interventions", [])
    centros = protocolo.get("contactsLocationsModule", {}).get("locations", [])

    return {
        "nct_id": identificacion.get("nctId"),
        "titulo": identificacion.get("briefTitle"),
        "estado": estado.get("overallStatus"),
        "fase": ", ".join(diseno.get("phases", [])) or None,
        "patrocinador": patrocinador.get("name"),
        "tipo_patrocinador": patrocinador.get("class"),
        "intervenciones": ", ".join(i["name"] for i in intervenciones if i.get("name")) or None,
        "tipo_intervencion": ", ".join(sorted({i["type"] for i in intervenciones if i.get("type")})) or None,
        "n_pacientes": diseno.get("enrollmentInfo", {}).get("count"),
        "aleatorizado": info_diseno.get("allocation"),
        "enmascaramiento": info_diseno.get("maskingInfo", {}).get("masking"),
        "fecha_inicio": estado.get("startDateStruct", {}).get("date"),
        "fecha_fin_prevista": estado.get("primaryCompletionDateStruct", {}).get("date"),
        "paises": ", ".join(sorted({c["country"] for c in centros if c.get("country")})) or None,
        "n_centros": len(centros) or None,
        "tiene_resultados": estudio.get("hasResults"),
    }


pd.DataFrame([extraer_fila(estudio) for estudio in datos["studies"]])

,nct_id,titulo,estado,fase,patrocinador,tipo_patrocinador,intervenciones,tipo_intervencion,n_pacientes,aleatorizado,enmascaramiento,fecha_inicio,fecha_fin_prevista,paises,n_centros,tiene_resultados
0,NCT00875433,Afatinib (BIBW 2992) QTcF Trial in Patients Wi...,COMPLETED,PHASE2,Boehringer Ingelheim,INDUSTRY,BIBW 2992,DRUG,60,NON_RANDOMIZED,NONE,2009-03,2011-04,United Kingdom,4,True
1,NCT02667587,An Investigational Immuno-therapy Study of Tem...,COMPLETED,PHASE3,Bristol-Myers Squibb,INDUSTRY,"Nivolumab, Temozolomide, Radiotherapy, Nivolum...","DRUG, OTHER, RADIATION",716,RANDOMIZED,TRIPLE,2016-05-09,2020-12-22,"Australia, Austria, Belgium, Canada, Denmark, ...",123,True


### DESCARGAMOS TODOS LOS ESTUDIOS PÁGINA A PÁGINA

In [5]:
def descargar_estudios(enfermedad):
    estudios = []
    parametros = {"query.cond": enfermedad, "pageSize": ESTUDIOS_POR_PAGINA}

    while True:
        respuesta = requests.get(URL_API, params=parametros, timeout=30)
        respuesta.raise_for_status()
        pagina = respuesta.json()

        estudios.extend(pagina["studies"])
        print(f"Descargados {len(estudios)} estudios")

        if "nextPageToken" not in pagina:
            break
        parametros["pageToken"] = pagina["nextPageToken"]
        time.sleep(0.5)

    return estudios


estudios = descargar_estudios(ENFERMEDAD)
tabla = pd.DataFrame([extraer_fila(estudio) for estudio in estudios])

print(tabla.shape)
tabla.head(10)

Descargados 100 estudios
Descargados 200 estudios
Descargados 300 estudios
Descargados 400 estudios
Descargados 500 estudios
Descargados 600 estudios
Descargados 700 estudios
Descargados 800 estudios
Descargados 900 estudios
Descargados 1000 estudios
Descargados 1100 estudios
Descargados 1200 estudios
Descargados 1300 estudios
Descargados 1400 estudios
Descargados 1500 estudios
Descargados 1600 estudios
Descargados 1700 estudios
Descargados 1800 estudios
Descargados 1900 estudios
Descargados 2000 estudios
Descargados 2100 estudios
Descargados 2200 estudios
Descargados 2288 estudios
(2288, 16)


,nct_id,titulo,estado,fase,patrocinador,tipo_patrocinador,intervenciones,tipo_intervencion,n_pacientes,aleatorizado,enmascaramiento,fecha_inicio,fecha_fin_prevista,paises,n_centros,tiene_resultados
0,NCT00875433,Afatinib (BIBW 2992) QTcF Trial in Patients Wi...,COMPLETED,PHASE2,Boehringer Ingelheim,INDUSTRY,BIBW 2992,DRUG,60.0,NON_RANDOMIZED,NONE,2009-03,2011-04,United Kingdom,4.0,True
1,NCT02667587,An Investigational Immuno-therapy Study of Tem...,COMPLETED,PHASE3,Bristol-Myers Squibb,INDUSTRY,"Nivolumab, Temozolomide, Radiotherapy, Nivolum...","DRUG, OTHER, RADIATION",716.0,RANDOMIZED,TRIPLE,2016-05-09,2020-12-22,"Australia, Austria, Belgium, Canada, Denmark, ...",123.0,True
2,NCT01082926,Phase I Study of Cellular Immunotherapy for Re...,COMPLETED,PHASE1,City of Hope Medical Center,OTHER,"therapeutic allogeneic lymphocytes, aldesleuki...","BIOLOGICAL, OTHER, PROCEDURE",6.0,NON_RANDOMIZED,NONE,2011-03-22,2013-09-23,United States,1.0,False
3,NCT02808364,Personalized Cellular Vaccine for Recurrent Gl...,COMPLETED,PHASE1,Guangdong 999 Brain Hospital,OTHER,Personalized cellular vaccine,BIOLOGICAL,10.0,NA,NONE,2016-03-01,2017-10-31,China,1.0,False
4,NCT02092038,Preoperative Chemoradiation for Glioblastoma,COMPLETED,PHASE1,Tampa General Hospital,OTHER,"Temozolomide, partial brain irradiation, stere...","DRUG, PROCEDURE, RADIATION",1.0,NA,NONE,2014-11,2015-07,United States,1.0,False
5,NCT03501134,Quality of Life of Patients With Glioblastoma ...,COMPLETED,None,Duke University,OTHER,NovoTTF,DEVICE,20.0,None,None,2018-08-08,2021-06-03,United States,1.0,False
6,NCT02626364,Study of Crenolanib in Recurrent/Refractory Gl...,COMPLETED,PHASE2,"Arog Pharmaceuticals, Inc.",INDUSTRY,crenolanib,DRUG,11.0,NA,NONE,2016-04,2020-07,United States,1.0,False
7,NCT04222309,Laparoscopically Harvested Omental Free Tissue...,SUSPENDED,PHASE1,Northwell Health,OTHER,Laparoscopically harvested omental free flap,PROCEDURE,10.0,NA,NONE,2020-01-06,2027-01-31,United States,1.0,False
8,NCT07791264,FPS-ZM1 With or Without Dexamethasone for Cont...,NOT_YET_RECRUITING,PHASE1,City of Hope Medical Center,OTHER,"Biospecimen Collection, Computed Tomography, D...","DRUG, PROCEDURE",24.0,NON_RANDOMIZED,NONE,2027-02-17,2028-08-28,United States,1.0,False
9,NCT01331616,Optic Neuropathy in 10 Patients With Glioblast...,WITHDRAWN,EARLY_PHASE1,West Penn Allegheny Health System,OTHER,Bevacizumab (Avastin),DRUG,0.0,NA,NONE,2011-03,2012-07,None,NaN,False


### LIMPIAMOS LA TABLA

In [6]:
ESTADOS = {
    "RECRUITING": "Reclutando",
    "NOT_YET_RECRUITING": "Aún no recluta",
    "ENROLLING_BY_INVITATION": "Reclutando por invitación",
    "ACTIVE_NOT_RECRUITING": "Activo, sin reclutar",
    "COMPLETED": "Completado",
    "TERMINATED": "Terminado antes de tiempo",
    "SUSPENDED": "Suspendido",
    "WITHDRAWN": "Retirado",
    "UNKNOWN": "Desconocido",
}
TIPOS_PATROCINADOR = {
    "INDUSTRY": "Industria",
    "NIH": "NIH",
    "FED": "Gobierno de EE. UU.",
    "OTHER_GOV": "Otro gobierno",
    "NETWORK": "Red de investigación",
    "OTHER": "Universidad / hospital",
    "INDIV": "Individual",
    "UNKNOWN": "Desconocido",
}
ALEATORIZACION = {"RANDOMIZED": "Sí", "NON_RANDOMIZED": "No", "NA": "No aplica"}
ENMASCARAMIENTO = {
    "NONE": "Abierto",
    "SINGLE": "Simple ciego",
    "DOUBLE": "Doble ciego",
    "TRIPLE": "Triple ciego",
    "QUADRUPLE": "Cuádruple ciego",
}

tabla["estado"] = tabla["estado"].replace(ESTADOS)
tabla["fase"] = tabla["fase"].str.replace("EARLY_PHASE1", "Fase 1 temprana").str.replace("PHASE", "Fase ")
tabla["tipo_patrocinador"] = tabla["tipo_patrocinador"].replace(TIPOS_PATROCINADOR)
tabla["aleatorizado"] = tabla["aleatorizado"].replace(ALEATORIZACION)
tabla["enmascaramiento"] = tabla["enmascaramiento"].replace(ENMASCARAMIENTO)
tabla["tiene_resultados"] = tabla["tiene_resultados"].map({True: "Sí", False: "No"})

for columna in ["fecha_inicio", "fecha_fin_prevista"]:
    tabla[columna] = pd.to_datetime(tabla[columna], format="mixed", errors="coerce")

tabla = tabla.astype({"n_pacientes": "Int64", "n_centros": "Int64"})

tabla["fase_maxima"] = tabla["fase"].str.findall(r"\d").str[-1].astype(float)
tabla = (
    tabla.sort_values(["fase_maxima", "n_pacientes"], ascending=False, na_position="last")
    .drop(columns="fase_maxima")
    .reset_index(drop=True)
)

tabla.head(10)

,nct_id,titulo,estado,fase,patrocinador,tipo_patrocinador,intervenciones,tipo_intervencion,n_pacientes,aleatorizado,enmascaramiento,fecha_inicio,fecha_fin_prevista,paises,n_centros,tiene_resultados
0,NCT07546669,Efficacy of Zoster Vaccination in Glioblastoma...,Aún no recluta,Fase 4,"University Hospital, Essen",Universidad / hospital,Additional vaccination against shingles (herpe...,"BIOLOGICAL, OTHER",230,Sí,Simple ciego,2027-01-02,2029-12-31,None,<NA>,No
1,NCT03975829,Pediatric Long-Term Follow-up and Rollover Study,"Activo, sin reclutar",Fase 4,Novartis Pharmaceuticals,Industria,"dabrafenib, trametinib",DRUG,163,No aplica,Abierto,2019-11-04,2026-11-05,"Argentina, Australia, Belgium, Brazil, Canada,...",54,No
2,NCT00686725,Standard Temodal (Temozolomide) Regimen Versus...,Completado,Fase 4,Merck Sharp & Dohme LLC,Industria,"Temozolomide, Radiotherapy","DRUG, RADIATION",99,Sí,Abierto,2008-06-24,2011-09-28,None,<NA>,Sí
3,NCT05342883,GammaTile and Stupp in Newly Diagnosed GBM,"Activo, sin reclutar",Fase 4,"GT Medical Technologies, Inc.",Industria,"Surgical tumor resection, GammaTile radiation ...",DEVICE,61,No aplica,Abierto,2022-08-17,2027-12-01,United States,16,No
4,NCT05235737,The Assessment of Immune Response in Newly Dia...,Desconocido,Fase 4,Medical University of Silesia,Universidad / hospital,"Pembrolizumab, Pembrolizumab",DRUG,36,Sí,Abierto,2022-03-01,2025-06-01,Poland,1,No
5,NCT06625047,Comparing Telehealth and In-person Assessments...,Completado,Fase 4,Mayo Clinic,Universidad / hospital,"Assessment, Questionnaire Administration, Tele...","BEHAVIORAL, DRUG, OTHER",30,No aplica,Abierto,2024-10-07,2025-11-12,United States,1,No
6,NCT01756729,Post-approval Study of NovoTTF-100A in Recurre...,Terminado antes de tiempo,Fase 4,NovoCure Ltd.,Industria,NovoTTF-100A,DEVICE,13,No,Abierto,2012-12-01,2018-01-01,United States,12,No
7,NCT05900908,Post-operative Adjuvant Therapy w/wo GammaTile...,Retirado,Fase 4,"GT Medical Technologies, Inc.",Industria,Gamma Tile-Surgically Targeted Radiation Thera...,"DEVICE, RADIATION",0,Sí,Simple ciego,2025-05-01,2026-06-01,None,<NA>,No
8,NCT03970447,A Trial to Evaluate Multiple Regimens in Newly...,Reclutando,"Fase 2, Fase 3",Global Coalition for Adaptive Research,Universidad / hospital,"Temozolomide, Lomustine, Regorafenib, Radiatio...","BIOLOGICAL, DRUG, RADIATION",2250,Sí,Abierto,2019-07-30,2028-06-01,"Australia, Canada, France, Germany, Switzerlan...",63,No
9,NCT00304031,Radiation Therapy (RT) and Temozolomide (TMZ) ...,Completado,Fase 3,Radiation Therapy Oncology Group,Red de investigación,"Concurrent temozolomide, Concurrent radiation ...","DRUG, RADIATION",1173,Sí,Abierto,2006-01-01,2011-02-01,"Canada, United States",351,Sí


### MOSTRAMOS LA TABLA FINAL CON COLORES

In [7]:
VERDE = "#c8e6c9"
AMARILLO = "#fff3b0"
ROJO = "#f8c8c8"

COLORES_ESTADO = {
    "Completado": VERDE,
    "Reclutando": AMARILLO,
    "Reclutando por invitación": AMARILLO,
    "Aún no recluta": AMARILLO,
    "Activo, sin reclutar": AMARILLO,
    "Terminado antes de tiempo": ROJO,
    "Suspendido": ROJO,
    "Retirado": ROJO,
}


def color_titulo(fila):
    color = COLORES_ESTADO.get(fila["estado"])
    return [f"background-color: {color}" if columna == "titulo" and color else "" for columna in fila.index]


(
    tabla.head(FILAS_A_MOSTRAR)
    .style
    .apply(color_titulo, axis=1)
    .format(na_rep="—")
    .format({"fecha_inicio": "{:%Y-%m}", "fecha_fin_prevista": "{:%Y-%m}"}, na_rep="—")
    .set_caption("Ensayos clínicos en glioblastoma: los más avanzados y con más pacientes primero")
    .hide(axis="index")
)

nct_id,titulo,estado,fase,patrocinador,tipo_patrocinador,intervenciones,tipo_intervencion,n_pacientes,aleatorizado,enmascaramiento,fecha_inicio,fecha_fin_prevista,paises,n_centros,tiene_resultados
NCT07546669,Efficacy of Zoster Vaccination in Glioblastoma Patients,Aún no recluta,Fase 4,"University Hospital, Essen",Universidad / hospital,"Additional vaccination against shingles (herpes zoster), Standard of care treatment without additional vaccination","BIOLOGICAL, OTHER",230,Sí,Simple ciego,2027-01,2029-12,—,—,No
NCT03975829,Pediatric Long-Term Follow-up and Rollover Study,"Activo, sin reclutar",Fase 4,Novartis Pharmaceuticals,Industria,"dabrafenib, trametinib",DRUG,163,No aplica,Abierto,2019-11,2026-11,"Argentina, Australia, Belgium, Brazil, Canada, Czechia, Denmark, Finland, France, Germany, Israel, Italy, Japan, Netherlands, Russia, Spain, Sweden, United Kingdom, United States",54,No
NCT00686725,Standard Temodal (Temozolomide) Regimen Versus Standard Regimen Plus Early Postsurgery Temodal for Newly Diagnosed Glioblastoma Multiforme (Study P05572),Completado,Fase 4,Merck Sharp & Dohme LLC,Industria,"Temozolomide, Radiotherapy","DRUG, RADIATION",99,Sí,Abierto,2008-06,2011-09,—,—,Sí
NCT05342883,GammaTile and Stupp in Newly Diagnosed GBM,"Activo, sin reclutar",Fase 4,"GT Medical Technologies, Inc.",Industria,"Surgical tumor resection, GammaTile radiation therapy implantation, Stupp protocol (EBRT and Temozolamide)",DEVICE,61,No aplica,Abierto,2022-08,2027-12,United States,16,No
NCT05235737,The Assessment of Immune Response in Newly Diagnosed Glioblastoma Patients Treated With Pembrolizumab,Desconocido,Fase 4,Medical University of Silesia,Universidad / hospital,"Pembrolizumab, Pembrolizumab",DRUG,36,Sí,Abierto,2022-03,2025-06,Poland,1,No
NCT06625047,Comparing Telehealth and In-person Assessments in Glioma Patients Receiving Oral Chemotherapy,Completado,Fase 4,Mayo Clinic,Universidad / hospital,"Assessment, Questionnaire Administration, Telemedicine Visit, Temozolomide","BEHAVIORAL, DRUG, OTHER",30,No aplica,Abierto,2024-10,2025-11,United States,1,No
NCT01756729,Post-approval Study of NovoTTF-100A in Recurrent GBM Patients,Terminado antes de tiempo,Fase 4,NovoCure Ltd.,Industria,NovoTTF-100A,DEVICE,13,No,Abierto,2012-12,2018-01,United States,12,No
NCT05900908,Post-operative Adjuvant Therapy w/wo GammaTile + Systemic Therapy,Retirado,Fase 4,"GT Medical Technologies, Inc.",Industria,"Gamma Tile-Surgically Targeted Radiation Therapy (STaRT), Stereotactic Radiation Therapy","DEVICE, RADIATION",0,Sí,Simple ciego,2025-05,2026-06,—,—,No
NCT03970447,A Trial to Evaluate Multiple Regimens in Newly Diagnosed and Recurrent Glioblastoma,Reclutando,"Fase 2, Fase 3",Global Coalition for Adaptive Research,Universidad / hospital,"Temozolomide, Lomustine, Regorafenib, Radiation, Paxalisib, VAL-083, VT1021, Troriluzole, ADI-PEG 20, AZD1390, Tinostamustine","BIOLOGICAL, DRUG, RADIATION",2250,Sí,Abierto,2019-07,2028-06,"Australia, Canada, France, Germany, Switzerland, United States",63,No
NCT00304031,Radiation Therapy (RT) and Temozolomide (TMZ) in Treating Patients With Newly Diagnosed Glioblastoma or Gliosarcoma,Completado,Fase 3,Radiation Therapy Oncology Group,Red de investigación,"Concurrent temozolomide, Concurrent radiation therapy, 100mg/m2 adjuvant temozolomide days 1 to 5 of 28 day cycle, 75mg/m2 adjuvant temozolomide days 1-21 of 28 day cycle","DRUG, RADIATION",1173,Sí,Abierto,2006-01,2011-02,"Canada, United States",351,Sí
